In [ ]:
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

Esta seção é responsável por carregar a biblioteca e estabelecer a conexão com o banco de dados.

from langchain_community.utilities import SQLDatabase: Importa a classe SQLDatabase do LangChain, que facilita a interação com bancos de dados SQL.

`db = SQLDatabase.from_uri("sqlite:///Chinook.db")`: Cria uma instância do objeto SQLDatabase. O `from_uri` conecta-se ao banco de dados especificado.

`"sqlite:///Chinook.db"`: É o URI de conexão. Indica que o banco é SQLite e está no arquivo chamado Chinook.db no mesmo diretório do script.

In [ ]:
from langchain_community.utilities import SQLDatabase

db= SQLDatabase.from_uri("sqlite:///Chinook.db")

Esta seção utiliza `dataclass` e é crucial para o padrão de Injeção de Dependência (DI) usado pelo LangGraph.

* `@dataclass`: Um decorador que cria automaticamente métodos como `__init__`, tornando a classe simples de usar como um contêiner de dados.

* class `RunTimeContext`: Esta classe define quais recursos estarão disponíveis durante a execução do seu "gráfico" (workflow) no LangGraph.

* `db: SQLDatabase`: Declara que a classe RunTimeContext deve ter um atributo chamado db que é uma instância de SQLDatabase. Isso significa que a conexão com o banco (db do item 1) será "injetada" no contexto de qualquer ferramenta que precisar dela.


Entendendo os Decoradores (`@dataclass` e `@tool`)
Decoradores são funções que modificam ou aprimoram outras funções ou classes sem alterar o código delas diretamente.

O que é `@dataclass`?
* Função: Simplifica a criação de classes cujo principal objetivo é apenas armazenar dados.

* O que ele faz: Ao invocar @dataclass acima de RunTimeContext, o Python automaticamente gera métodos "boilerplate" (padrão) como o `__init__` (construtor) e o `__repr__` (como o objeto deve ser representado).

Sem o decorador: Você teria que escrever o construtor manualmente:

~~~Python
class RunTimeContext:
    def __init__(self, db: SQLDatabase):
        self.db = db
~~~

Com o decorador: Você só precisa declarar os atributos:

~~~Python
@dataclass
class RunTimeContext:
    db: SQLDatabase  # O construtor é gerado automaticamente!
~~~


O que é `@tool`?

~~~Python
@tool
def execute_sql(query: str) -> str:
    # ...
~~~

* Função: Transforma uma função Python comum em um formato que é compreendido por modelos de IA (como GPT-4, Claude, etc.).

O que ele faz:

* Ele empacota a função execute_sql.

* Ele usa o nome da função (execute_sql) e a docstring (o texto entre aspas triplas """...""") para gerar um esquema JSON (uma descrição estruturada).

* Quando o modelo de IA precisa de informação do banco, ele vê esse esquema e diz: "Eu preciso chamar a ferramenta execute_sql com o argumento query."

In [ ]:
from dataclasses import dataclass
from langchain_community.utilities import SQLDatabase

# Defina o contexto para suportar a injeção de dependência.
@dataclass
class RunTimeContext:
    db: SQLDatabase

Esta é a função principal que a IA usará para interagir com o banco de dados, transformando a interação SQL em uma ferramenta LangChain.

* from `langchain_core.tools import tool`: Importa o decorador que transforma uma função Python simples em uma ferramenta que um modelo de linguagem (LLM) pode usar.

* `@tool`: Aplica o decorador. A docstring ("""Execute a SQLite command...""") é usada pelo LLM para entender o que a ferramenta faz.

* `def execute_sql(query: str) -> str:`: A função recebe uma string (query) que será o comando SQL (como SELECT * FROM tracks LIMIT 5).

* `runtime = get_runtime(RunTimeContext)`: Esta é a parte central da Injeção de Dependência. Ela obtém o objeto runtime do LangGraph, que armazena o contexto definido na Seção 2.

* db = `runtime.context.db:` Acessa a instância do SQLDatabase que foi colocada (injetada) no contexto (db da Seção 1).

* `return db.run(query)`: Executa o comando SQL (query) no banco de dados. O método db.run() já é otimizado pelo LangChain para formatar os resultados.

* `except Exception as e:` : Adiciona tratamento de erro. Se houver um problema (por exemplo, a query SQL estiver mal formatada), ele retorna a mensagem de erro em vez de travar o programa.

**Nota de segurança: Esta demonstração não inclui um filtro para comandos gerados pelo LLM. Em um ambiente de produção, você deve limitar o escopo dos comandos gerados pelo LLM.**



Explicação mais específica de cada uma dessas linhas

O que faz `runtime = get_runtime(RunTimeContext)`?

~~~Python

runtime = get_runtime(RunTimeContext)

~~~

* Função: É a forma de a função execute_sql pedir ao LangGraph (o motor de execução) que lhe forneça o ambiente em que ela está rodando.

* O que ele faz: Ele busca o objeto de execução (runtime) atual e, mais importante, garante que ele tenha o contexto que você definiu (RunTimeContext).

* Tradução: "Ei, sistema de execução, me dê o objeto que contém todos os recursos que eu preciso (incluindo a conexão com o banco) conforme definido na minha RunTimeContext."

O que faz `db = runtime.context.db`?

~~~Python

db = runtime.context.db
~~~

* Função: Acessa o recurso que foi injetado (o objeto SQLDatabase) a partir do ambiente de execução.

O que ele faz:

* runtime: O objeto que representa o ambiente de execução atual.

* .context: Acessa a parte do runtime que contém os recursos definidos (onde o RunTimeContext foi injetado).

* .db: Acessa o atributo db do contexto, que é a instância real e ativa da conexão com o banco de dados (SQLDatabase).


O que faz `db.run(query)`?

~~~Python

return db.run(query)
~~~ 
* Função: Executa o comando SQL no banco de dados.

O que ele faz: O método run() é um utilitário do SQLDatabase do LangChain. Ele recebe a string SQL e:

* Abre uma conexão (se não estiver aberta).

* Executa o comando SQL.

* Lê os resultados.

* Fecha a conexão.

* Formata os resultados como uma string legível para ser retornada ao modelo de IA.

In [ ]:
from langchain_core.tools import tool
from langgraph.runtime import get_runtime

@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite command and return results"""
    runtime= get_runtime(RunTimeContext)
    db= runtime.context.db

    try:
        return db.run(query)
    
    except  Exception as e:
        return f"Error: {e}"

In [ ]:
SYSTEM_PROMPT= """
Rules:
- Think step-by-step.
- When you need data, call the tool `execute_sql` with ONE SELECT query.
- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.
- Limit to 5 rows of output unless the user explicitly asks otherwise.
- If the tool returns 'Error:', revise the SQL and try again.
- Prefer explicit column lists; avoid SELECT *.
"""

In [ ]:
from langchain.agents import create_agent

agent= create_agent(
    model= "openai:gpt-5",
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RunTimeContext
)

In [ ]:
question= "Which table has the largest number of entries?"

for step in agent.stream(
    {"messages": question},
    context= RunTimeContext(db=db),
    stream_mode="values"
):
    step["messages"][-1].pretty_prnt()


In [ ]:
question = "Which genre on average has the longest tracks?"

for step in agent.stream(
    {"messages": question},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

In [ ]:
question = "Please list all of the tables"

for step in agent.stream(
    {"messages": question},
    context={"db": db},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

In [ ]:
question = "TRY YOUR OWN QUERY HERE"

for step in agent.stream(
    {"messages": question},
    context={"db": db},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()